In [1]:
# !wget http://mattmahoney.net/dc/text8.zip
# !unzip text8.zip

In [2]:
from collections import Counter
import numpy as np

In [3]:
with open('text8') as file:
    text = file.read()

tokens = text.split()

# get counts of all words in dataset
MIN_WORD_COUNT = 4
counts = Counter(tokens)
counts = {w: c for w,c in counts.items() if c >= MIN_WORD_COUNT and len(w) > 1}

# get vocabulary sorted desc by word frequency
vocab = sorted(counts, key=counts.get, reverse=True)
VOCAB_SIZE = len(vocab)

# build words <-> index relations and convert tokens to those indexes
word2idx = {w: i for i,w in enumerate(vocab)}
idx2word = {i: w for w,i in word2idx.items()}
token_ids = [word2idx[token] for token in tokens if token in word2idx]

print(f"Vocab size: {VOCAB_SIZE:,}")
print(f"Corpus length: {len(token_ids):,}")

Vocab size: 82,270
Corpus length: 16,130,560


In [4]:
# get shifted probabilities of word occurances
prob = np.array([counts[idx2word[i]] ** 0.75 for i in range(VOCAB_SIZE)])
prob /= prob.sum()

# create lookup-table for negative samples with those probabilities
NEG_TABLE_SIZE = 1_000_000
neg_table = np.random.choice(VOCAB_SIZE, size=NEG_TABLE_SIZE, p=prob)

In [5]:
# SGNS training loop
WINDOW_SIZE = 5
NEGATIVE_SAMPLES = 5
EMBED_DIM = 200

EPOCHS = 1
LR = 0.025

total = len(token_ids)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Target & Context embedding matrices init
W = np.random.rand(VOCAB_SIZE, EMBED_DIM)
C = np.random.rand(VOCAB_SIZE, EMBED_DIM)

for epoch in range(EPOCHS):
    for i, center_id in enumerate(token_ids):

        # compute dynamic window for a given center token
        # and get context token ids for positive samples

        window_size = np.random.randint(1, WINDOW_SIZE + 1)
        
        left  = max(0, i - window_size)
        right = min(i + window_size, len(token_ids) - 1)
        
        context_ids = token_ids[left:i] + token_ids[i+1:right+1]

        # for each positive samples get negative samples
        # from lookup-table of random words

        for positive in context_ids:
            negative_ids = np.random.randint(0, VOCAB_SIZE, size=NEGATIVE_SAMPLES)
            negatives = neg_table[negative_ids]

            v = W[center_id] # (EMBED_DIM,)
            u_pos = C[positive] # (EMBED_DIM,)
            u_neg = C[negatives] # (NEGATIVE_SAMPLES, EMBED_DIM)

            # forward pass
            pos_align = sigmoid(v @ u_pos)
            neg_aligns = sigmoid(u_neg @ v)

            # gradients
            grad_pos = (pos_align - 1) * v
            grad_neg = neg_aligns[:, None] * v
            grad_v = (pos_align - 1) * u_pos + neg_aligns @ u_neg

            # backward pass
            W[center_id] -= LR * grad_v
            C[positive]  -= LR * grad_pos
            C[negatives] -= LR * grad_neg
        
        if i % 500_000 == 0:
            print(f"epoch {epoch+1} | {i:,}/{total:,}")


epoch 1 | 0/16,130,560
epoch 1 | 500,000/16,130,560
epoch 1 | 1,000,000/16,130,560
epoch 1 | 1,500,000/16,130,560
epoch 1 | 2,000,000/16,130,560
epoch 1 | 2,500,000/16,130,560
epoch 1 | 3,000,000/16,130,560
epoch 1 | 3,500,000/16,130,560
epoch 1 | 4,000,000/16,130,560
epoch 1 | 4,500,000/16,130,560
epoch 1 | 5,000,000/16,130,560
epoch 1 | 5,500,000/16,130,560
epoch 1 | 6,000,000/16,130,560
epoch 1 | 6,500,000/16,130,560
epoch 1 | 7,000,000/16,130,560
epoch 1 | 7,500,000/16,130,560
epoch 1 | 8,000,000/16,130,560
epoch 1 | 8,500,000/16,130,560
epoch 1 | 9,000,000/16,130,560
epoch 1 | 9,500,000/16,130,560
epoch 1 | 10,000,000/16,130,560
epoch 1 | 10,500,000/16,130,560
epoch 1 | 11,000,000/16,130,560
epoch 1 | 11,500,000/16,130,560
epoch 1 | 12,000,000/16,130,560
epoch 1 | 12,500,000/16,130,560
epoch 1 | 13,000,000/16,130,560
epoch 1 | 13,500,000/16,130,560
epoch 1 | 14,000,000/16,130,560
epoch 1 | 14,500,000/16,130,560
epoch 1 | 15,000,000/16,130,560
epoch 1 | 15,500,000/16,130,560
epoch 

In [6]:
def get_neighbours(word: str, k: int = 5) -> list[str]:
    i = word2idx[word]
    target_embed = W[i]

    target_norm = np.linalg.norm(target_embed)

    dots = W @ target_embed
    norms = np.linalg.norm(W, axis=1)

    similarities = dots / (norms * target_norm + 1e-9)

    topk = np.argsort(similarities)[::-1][1:k+1]
    return [(idx2word[i], round(float(similarities[i]), 4)) for i in topk]

In [ ]:
get_neighbours("king")

[('son', 0.6779),
 ('vi', 0.6598),
 ('vii', 0.655),
 ('elizabeth', 0.6407),
 ('henry', 0.6365)]

In [8]:
get_neighbours("poland")

[('hungary', 0.7402),
 ('russia', 0.6874),
 ('austria', 0.6661),
 ('finland', 0.6645),
 ('germany', 0.6561)]

In [10]:
get_neighbours("queen")

[('elizabeth', 0.6868),
 ('son', 0.6647),
 ('wife', 0.6241),
 ('catherine', 0.6218),
 ('mary', 0.6211)]